In [61]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchResults
import requests
from langchain_classic import hub
from langsmith import Client
from langchain_classic.agents import AgentExecutor, create_react_agent
import re


In [62]:
llm = ChatOpenAI()

In [63]:
webSearchtool = DuckDuckGoSearchResults()


@tool 
def calculateOld(a: int , b: int, op: str) -> int:
    "performs a (operator) b"
    if(op == '+'):
        return a + b;
    if(op == '-'):
        return a - b;
    if(op == '*'):
        return a * b;
    if(op == '/'):
        return a / b;

    return a + b;

@tool
def calculate(expression: str) -> str:
    """Evaluate an arithmetic expression. Input must be digits and + - * / ( ) only, e.g. '241000000 + 128000000'."""
    if not re.fullmatch(r"[\d+\-*/(). ]+", expression):
        return "Error: numbers and + - * / ( ) only. Do not pass words."
    return str(eval(expression))



In [64]:
webSearchtool.invoke("latest news in india")
# Step 2: Pull the ReAct prompt from LangChain Hub
inputPrompt = Client().pull_prompt("hwchase17/react", dangerously_pull_public_prompt=True)


In [65]:
agent = create_react_agent(
llm=llm,
prompt = inputPrompt,
tools=[webSearchtool, calculate]
)

In [66]:
agentExecutor = AgentExecutor(
    agent = agent,
    tools = [webSearchtool, calculate],
    verbose= True
)

In [67]:
response = agentExecutor.invoke({"input": "Across all states in india, calculate the population"})




> Entering new AgentExecutor chain...
I should use a search engine to find the population of all states in India and then calculate the total population.
Action: duckduckgo_results_json
Action Input: "population of all states in India"snippet: Under the Indian Constitution and laws, the states of India are self-governing administrative divisions, each having a state government. The legal power to manage affairs in each state is shared or divided between the particular state government on one hand and the national union government on the other. The union territories are directly governed by the union government; no state level ..., title: States and union territories of India - Wikipedia, link: https://en.wikipedia.org/wiki/States_and_union_territories_of_India, snippet: Just 1% of India's population lives in 12 least populous states or union territories, while 10% of the country's population lives in 21 least populous states or union territories. The largest state, Uttar Pradesh, has

In [69]:
response

{'input': 'Across all states in india, calculate the population',
 'output': 'The total population across all states in India is 2,917,900,000.'}